# Demosat Data Utils Smoke Test - Spec #11 EHA/EVR Exit-Condition Gates

This notebook exercises the DataFrame layer for EHA and EVR as defined in spec #11.

In [ ]:
from demosat_data_utils.eha import DemosatChannelFrame
from demosat_data_utils.evr import DemosatEvrFrame, EvrItem, EvrContainer
print('Public imports OK')

In [ ]:
import pandas as pd

eha_path = 'data/sim_eha_sample.csv'
evr_path = 'data/sim_evr_sample.csv'

eha = DemosatChannelFrame(csv_path=eha_path, coerce=True)
evr = DemosatEvrFrame(csv_path=evr_path, coerce=True)

print('EHA loaded:', len(eha), 'rows')
print('EVR loaded:', len(evr), 'rows')

In [ ]:
# EHA column retention and semantics
expected_eha_cols = ['recordType','sessionId','sessionHost','channelId','dssId','vcid','name','module','ert','scet','rct','lst','sclk','dn','dnStr','eu','status','dnAlarmState','euAlarmState','realtime','type']
assert all(col in eha.columns for col in expected_eha_cols), 'Missing EHA columns'
assert pd.api.types.is_datetime64_any_dtype(eha['scet']), 'scet not datetime'
assert eha.LABEL_COL == 'channelId'
assert eha.VALUE_COL == 'eu'
assert eha.DEFAULT_TIME_LABEL == 'scet'
print('EHA columns, parsing, semantics OK')

In [ ]:
# EHA alarm styling
styles = [eha.iloc[i].default_html_row_style for i in range(len(eha))]
assert styles[0] == {}, 'row0 should be neutral'
assert styles[1] == {'background-color': '#FFF3CC'}, 'row1 yellow'
assert styles[3] == {'background-color': '#FFCCCC'}, 'row3 red'
assert styles[5] == {'background-color': '#FFCCCC'}, 'row5 red precedence'
assert styles[6] == {'background-color': '#FFCCCC'}, 'row6 case-insensitive RED'
print('EHA alarm styling OK')

In [ ]:
# EVR column retention and semantics
expected_evr_cols = ['recordType','sessionId','sessionHost','name','module','level','eventId','vcid','dssId','fromSse','realtime','sclk','scet','ert','rct','lst','message','metadataKeywordList','metadataValuesList','metadata']
assert all(col in evr.columns for col in expected_evr_cols), 'Missing EVR columns'
assert 'metadata' in evr.columns
assert pd.api.types.is_datetime64_any_dtype(evr['scet'])
assert evr.LABEL_COL == 'name'
assert evr.VALUE_COL == 'message'
assert evr.DEFAULT_TIME_LABEL == 'scet'
print('EVR columns, parsing, semantics OK')

In [ ]:
# EVR level styling
from demosat_data_utils.evr import EVR_LEVEL_COLORS
levels = ['DIAGNOSTIC','COMMAND','ACTIVITY_LO','ACTIVITY_HI','WARNING_LO','WARNING_HI','FATAL','SIM_ERROR']
for lvl in levels:
    # Use existing rows if available, else create dummy
    sample = evr[evr['level']==lvl]
    if len(sample) > 0:
        row = sample.iloc[0]
    else:
        row = DemosatEvrFrame([{'name':'X','message':'m','level':lvl,'scet':'2024-001T00:00:00.000000'}]).iloc[0]
    style = row.default_html_row_style
    assert style, f'No style for level {lvl}'
    if lvl == 'SIM_ERROR':
        assert style == EVR_LEVEL_COLORS[lvl]
# Unknown level
unknown = DemosatEvrFrame([{'name':'X','message':'m','level':'UNKNOWN','scet':'2024-001T00:00:00.000000'}])
assert unknown.iloc[0].default_html_row_style == {}, 'unknown level should be safe'
print('EVR level styling OK')

In [ ]:
# EVR filter_level preserves order and type
filtered = evr.filter_level('COMMAND')
assert isinstance(filtered, DemosatEvrFrame)
assert len(filtered) == 2  # sample has two COMMAND rows
multi = evr.filter_level(['ACTIVITY_LO','COMMAND'])
assert isinstance(multi, DemosatEvrFrame)
assert len(multi) == 3  # ACTIVITY_LO + 2 COMMAND
# Order preserved: first row should be the first ACTIVITY_LO or COMMAND in original
assert list(multi['level'])[:2] == ['ACTIVITY_LO','COMMAND']
print('EVR filter_level OK')

In [ ]:
# EHA lad() test
rows = [
    {'channelId':'A','eu':1,'scet':'2024-001T00:00:01.000000','dnAlarmState':None,'euAlarmState':None},
    {'channelId':'A','eu':2,'scet':'2024-001T00:00:03.000000','dnAlarmState':None,'euAlarmState':None},
    {'channelId':'B','eu':5,'scet':'2024-001T00:00:02.000000','dnAlarmState':None,'euAlarmState':None},
    {'channelId':'A','eu':1.5,'scet':'2024-001T00:00:02.000000','dnAlarmState':None,'euAlarmState':None},
]
df = DemosatChannelFrame(rows)
lad_df = df.lad()
assert isinstance(lad_df, DemosatChannelFrame)
vals = dict(zip(lad_df['channelId'], lad_df['eu']))
assert vals['A']==2 and vals['B']==5
print('EHA lad() OK')

In [ ]:
# Type preservation
eha_copy = eha.copy()
assert isinstance(eha_copy, DemosatChannelFrame)
eha_sorted = eha.sort_values('scet')
assert isinstance(eha_sorted, DemosatChannelFrame)
row_loc = eha.loc[0]
assert isinstance(row_loc, eha.ROW_SERIES_CLASS)
row_iloc = eha.iloc[0]
assert isinstance(row_iloc, eha.ROW_SERIES_CLASS)
eha_filtered = eha[eha['channelId']=='ADCS-0001']
assert isinstance(eha_filtered, DemosatChannelFrame)
print('Type preservation OK')

In [ ]:
# Legacy imports unchanged
assert EvrItem.NAME == 'EVR'
assert EvrContainer.DATA_ITEM_CLS is EvrItem
expected_levels = ['DIAGNOSTIC','COMMAND','ACTIVITY_LO','ACTIVITY_HI','WARNING_LO','WARNING_HI','FATAL','SIM_ERROR']
assert EvrContainer.LEVELS == expected_levels
print('Legacy EvrContainer/EvrItem OK')